<a href="https://colab.research.google.com/github/yasirusman85/Training-2/blob/main/Copy_of_Training_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
lukechugh_best_alzheimer_mri_dataset_99_accuracy_path = kagglehub.dataset_download('lukechugh/best-alzheimer-mri-dataset-99-accuracy')

print('Data source import complete.')


100%|██████████| 71.5M/71.5M [00:00<00:00, 103MB/s] 

Extracting files...


Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Training Folders (Balanced):

Each of the 4 classes in the train directory contains exactly 2,560 images.
Testing Folders (Unbalanced):

No Impairment: 640 images
Very Mild Impairment: 448 images
Mild Impairment: 179 images
Moderate Impairment: 12 images

In [5]:
import os

# Define the base directory
base_dir = lukechugh_best_alzheimer_mri_dataset_99_accuracy_path

print(f"Exploring dataset at: {base_dir}\n")

# Recursively find directories that actually contain images
for root, dirs, files in os.walk(base_dir):
    # Filter for image files
    images = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if images:
        # If this folder contains images, treat it as a class folder
        folder_name = os.path.relpath(root, base_dir)
        print(f"Folder: {folder_name}")
        print(f" - Number of images: {len(images)}")
        print(f" - Sample images: {images[:3]}\n")

Exploring dataset at: /root/.cache/kagglehub/datasets/lukechugh/best-alzheimer-mri-dataset-99-accuracy/versions/1

Folder: Combined Dataset/train/Moderate Impairment
 - Number of images: 2560
 - Sample images: ['ModerateImpairment (2017).jpg', 'ModerateImpairment (1259).jpg', 'ModerateImpairment (2488).jpg']

Folder: Combined Dataset/train/Mild Impairment
 - Number of images: 2560
 - Sample images: ['MildImpairment (1868).jpg', 'MildImpairment (581).jpg', 'MildImpairment (966).jpg']

Folder: Combined Dataset/train/No Impairment
 - Number of images: 2560
 - Sample images: ['NoImpairment (363).jpg', 'NoImpairment (1952).jpg', 'NoImpairment (2369).jpg']

Folder: Combined Dataset/train/Very Mild Impairment
 - Number of images: 2560
 - Sample images: ['VeryMildImpairment (96).jpg', 'VeryMildImpairment (1710).jpg', 'VeryMildImpairment (1321).jpg']

Folder: Combined Dataset/test/Moderate Impairment
 - Number of images: 12
 - Sample images: ['17.jpg', '17 (2).jpg', '14.jpg']

Folder: Combined 

### Step 1: Create a Balanced Subset
We are creating a new directory and copying exactly 1,000 random images from each class of the training set to ensure the model trains on a balanced distribution.

In [6]:
import os
import shutil
import random

# Source and Destination paths
src_train_dir = os.path.join(lukechugh_best_alzheimer_mri_dataset_99_accuracy_path, 'Combined Dataset/train')
subset_train_dir = '/content/balanced_train_subset'

# Create destination directory
os.makedirs(subset_train_dir, exist_ok=True)

# Selection parameters
IMAGES_PER_CLASS = 1000

# Iterate through each category
for category in os.listdir(src_train_dir):
    src_cat_path = os.path.join(src_train_dir, category)
    dest_cat_path = os.path.join(subset_train_dir, category)

    if os.path.isdir(src_cat_path):
        os.makedirs(dest_cat_path, exist_ok=True)

        # Get all image files
        all_images = [f for f in os.listdir(src_cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # Randomly sample 1000 images
        selected_images = random.sample(all_images, min(len(all_images), IMAGES_PER_CLASS))

        # Copy to the new directory
        for img in selected_images:
            shutil.copy(os.path.join(src_cat_path, img), os.path.join(dest_cat_path, img))

        print(f"Copied {len(selected_images)} images for class: {category}")

Copied 1000 images for class: Moderate Impairment
Copied 1000 images for class: Mild Impairment
Copied 1000 images for class: No Impairment
Copied 1000 images for class: Very Mild Impairment


### Step 2: Data Augmentation
Now we apply `ImageDataGenerator` to the balanced subset. This will artificially increase our dataset variety by applying transformations like rotation, zooming, and flipping during training.

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the augmentation parameters
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Create the generator using the balanced subset
train_generator = train_datagen.flow_from_directory(
    subset_train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

print("Data augmentation generator is ready with 1,000 images per class.")

Found 4000 images belonging to 4 classes.
Data augmentation generator is ready with 1,000 images per class.


### Step 3: Build the DenseNet121 Model
We initialize the DenseNet121 model with pre-trained ImageNet weights, freeze the base layers to preserve the learned features, and add a custom classification head for our 4 classes.

In [9]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Load the pre-trained DenseNet121
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers
base_model.trainable = False

# Build the custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)  # Added dropout for regularization
predictions = Dense(4, activation='softmax')(x)

# Construct the final model
model = Model(inputs=base_model.input, outputs=predictions)

model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_2    │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d_2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_3    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_3… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,564,356 (28.86 MB)

 Trainable params: 526,852 (2.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

### Step 4: Compile and Train
We'll use the Adam optimizer and Categorical Crossentropy loss. Since you are using a T4 GPU, training should be relatively fast for 4,000 images.

In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Start training
print("Starting training on T4 GPU...")
history = model.fit(
    train_generator,
    epochs=10,
    verbose=1
)

Starting training on T4 GPU...
Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 98s 503ms/step - accuracy: 0.4545 - loss: 1.2477
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 65s 518ms/step - accuracy: 0.5785 - loss: 0.9509
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 505ms/step - accuracy: 0.6043 - loss: 0.8957
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 500ms/step - accuracy: 0.6288 - loss: 0.8375
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 501ms/step - accuracy: 0.6467 - loss: 0.8107
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 498ms/step - accuracy: 0.6555 - loss: 0.7939
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 497ms/step - accuracy: 0.6568 - loss: 0.7620
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 497ms/step - accuracy: 0.6683 - loss: 0.7502
Epoch 9/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 501ms/step - accuracy: 0.6702 - loss: 0.7519
Epoch 10/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 489ms/step - accuracy: 0.6765 - loss: 0.7369


### Step 5: Evaluate on Test Data
We'll prepare the test data using the same rescaling as the training set and evaluate the model's performance.

In [11]:
import os

# Path to test data
test_dir = os.path.join(lukechugh_best_alzheimer_mri_dataset_99_accuracy_path, 'Combined Dataset/test')

# Create test generator
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Evaluate the model
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Test Loss: {loss:.4f}")

Found 1279 images belonging to 4 classes.
40/40 ━━━━━━━━━━━━━━━━━━━━ 33s 469ms/step - accuracy: 0.4590 - loss: 1.1918
Test Accuracy: 45.90%
Test Loss: 1.1918


### Step 6: Save the Model
Saving the model to the local Colab environment.

In [12]:
# Save the model in Keras format
model_save_path = 'alzheimer_densenet_model.keras'
model.save(model_save_path)
print(f"Model saved successfully to {model_save_path}")

Model saved successfully to alzheimer_densenet_model.keras


In [13]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the destination path in the 'Aggregation' folder
drive_save_path = '/content/drive/MyDrive/Aggregation'
os.makedirs(drive_save_path, exist_ok=True)

# Copy the model to the specified Drive folder
shutil.copy('alzheimer_densenet_model.keras', os.path.join(drive_save_path, 'alzheimer_densenet_model.keras'))

print(f"Model successfully backed up to Google Drive at: {drive_save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model successfully backed up to Google Drive at: /content/drive/MyDrive/Aggregation
